<a href="https://colab.research.google.com/github/skalamin65/Text-summarizer-app/blob/main/text_summarizing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [ ]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [ ]:
import re
def clean_data(text):
  text=re.sub(r"\r\n"," ",text) # line
  text=re.sub(r"\s+"," ",text) # spaces
  text=re.sub(r"<.*?>"," ",text) # html tags
  text=text.strip().lower()
  return text

In [ ]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [ ]:
train_data.head()

,id,dialogue,summary
0,13811908,violet: hi! i came across this austin's articl...,violet sent claire austin's article.
1,13716431,pat: so does anyone know when the stream is go...,pat and lou are waiting for the stream but kev...
2,13810214,jane: jane: whaddya think? shona: this ur ti...,jane is updating her tinder profile tonight an...
3,13729823,"adam: do u have a map of paris? tom: yes, why?...",tom has a map of paris.
4,13681400,"frank: hi, how's the family? mike: great! sam'...","mike is happy, because sam's moved out. mike a..."


In [ ]:
Tokenizer=T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
def tokenize(data):
  inputs=Tokenizer(data["dialogue"],max_length=512,padding="max_length",truncation=True)
  targets=Tokenizer(data["summary"],max_length=150,padding="max_length",trunction=True)
  inputs["labels"]=targets["input_ids"]
  return inputs

In [ ]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [ ]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
import torch
if torch.cuda.is_available():
  device=torch.device("cuda")
else:
  device=torch.device("cpu")
print("device: ",device)
model.to(device)


device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
# Training Arguments
training_args=TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500
)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.648336,0.381535
2,0.397009,0.361645
3,0.373905,0.354698
4,0.361505,0.350491
5,0.355502,0.349725
6,0.351459,0.349056


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9146192932128906, metrics={'train_runtime': 1375.2688, 'train_samples_per_second': 17.451, 'train_steps_per_second': 2.181, 'total_flos': 3248203235328000.0, 'train_loss': 0.9146192932128906, 'epoch': 6.0})

In [ ]:
model.save_pretrained("/content/drive/MyDrive/saved_summary_model")
Tokenizer.save_pretrained("/content/drive/MyDrive/saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/saved_summary_model/tokenizer_config.json',
 '/content/drive/MyDrive/saved_summary_model/tokenizer.json')

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("/content/drive/MyDrive/saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
# Test the core logic
def summarize_dialouge(dialogue):
  dialouge=clean_data(dialogue)

  # Tokenize
  inputs=Tokenizer(
      dialouge,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"
  ).to(device)

  # generate the summary => token ids
  model.to(device)
  targets=model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=150,
      num_beams=4,
      early_stopping=True
  ).to(device)

  # decode our output
  summary=Tokenizer.decode(targets[0],skip_special_tokens=True)
  return summary

In [ ]:
test_dialouge="""
Sarah: Hey everyone, are we still on for the cabin trip this weekend?
Mark: Yes! I'm bringing the snacks and drinks. Do we need to bring our own bedding?
Emily: I think the cabin has sheets, but we should bring extra blankets just in case. It gets pretty cold at night up in the mountains.
Sarah: Good call. What time is everyone heading up? I plan to leave around 3 PM on Friday.
Mark: I can't leave until after work, so I'll probably be on the road by 6 PM.
Emily: I'll ride up with Sarah at 3 PM so we can get the groceries sorted and the place warmed up before Mark arrives.
Mark: Sounds like a plan. Just send me the address again, I can't find it in the chat history.
Sarah: Sure, I'll text it to you right now. See you guys Friday!
"""
summary=summarize_dialouge(test_dialouge)
print(summary)

sarah is on for the cabin trip this weekend. emily is bringing snacks and drinks. sarah will be on the road by 6 pm on friday.
